In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


def scatter_with_rmse(obs: xr.DataArray, sim: xr.DataArray, title: str, ax=None) -> float:
    """
    Align obs/sim, drop NaNs, plot a scatter with a 1:1 line, and return RMSE.
    """
    obs_aligned, sim_aligned = xr.align(obs, sim, join="inner")
    obs_flat = obs_aligned.values.ravel()
    sim_flat = sim_aligned.values.ravel()

    mask = ~np.isnan(obs_flat) & ~np.isnan(sim_flat)
    obs_flat, sim_flat = obs_flat[mask], sim_flat[mask]

    if len(obs_flat) == 0:
        raise ValueError(f"No overlapping, non-NaN data points for '{title}'.")

    value = float(np.sqrt(np.mean((obs_flat - sim_flat) ** 2)))

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(obs_flat, sim_flat, s=10, alpha=0.4, edgecolor="none")

    lo = min(obs_flat.min(), sim_flat.min())
    hi = max(obs_flat.max(), sim_flat.max())
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, label="1:1 line")

    ax.set_xlabel("Observed")
    ax.set_ylabel("Simulated")
    ax.set_title(f"{title}\nRMSE = {value:.4f}   n = {len(obs_flat)}")
    ax.legend()
    ax.set_aspect("equal", adjustable="box")
    return value


# ---------------------------------------------------------------------
# LOAD YOUR DATA HERE
# ---------------------------------------------------------------------
# Replace these lines with however you actually load/select your arrays.
# Example if reading from NetCDF files:
#
# moisture_obs = xr.open_dataarray("moisture_obs.nc")
# moisture_sim = xr.open_dataarray("moisture_sim.nc")
# temp_obs     = xr.open_dataarray("temp_obs.nc")
# temp_sim     = xr.open_dataarray("temp_sim.nc")

site = "TVC"
ctsm_file = f"..\\CTSM_data\\run_1\\merged_files\\{site}_1995-2014.nc"
df = xr.open_dataset(ctsm_file, engine="netcdf4")

tsoi_ctsm = df["TSOI"]
tsoi_ctsm = tsoi_ctsm.resample(time="1D").mean()
ds = xr.open_dataset(f"../Observational_data/{site}_obs/processed/{site}_TSOI.nc")
tsoi_obs = ds["TSOI"]
tsoi_obs = tsoi_obs.sortby("depth")
tsoi_obs = tsoi_obs.resample(time="1D").mean()

h2osoi_ctsm = df["H2OSOI"]
h2osoi_ctsm = h2osoi_ctsm.resample(time="1D").mean()
ds = xr.open_dataset(f"../Observational_data/{site}_obs/processed/{site}_H2OSOI.nc")
h2osoi_obs = ds["H2OSOI"]
h2osoi_obs = h2osoi_obs.sortby("depth")
h2osoi_obs = h2osoi_obs.resample(time="1D").mean()

moisture_obs = h2osoi_obs
moisture_sim = h2osoi_ctsm
temp_obs = tsoi_obs
# temp_sim is loaded for completeness but not required for the analyses below


C:\Users\madel\AppData\Local\Temp\ipykernel_14468\2216161611.py:94: FutureWarning: In a future version, xarray will not decode the variable 'SNOW_PERSISTENCE' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  df = xr.open_dataset(ctsm_file, engine="netcdf4")


In [3]:
print("moisture_obs:", moisture_obs.dims, moisture_obs.shape)
print("moisture_sim:", moisture_sim.dims, moisture_sim.shape)
print(moisture_sim.coords)

moisture_obs: ('time', 'depth') (7300, 2)
moisture_sim: ('time', 'levsoi', 'lndgrid') (7300, 20, 1)
Coordinates:
  * time     (time) object 58kB 1995-01-01 00:00:00 ... 2014-12-31 00:00:00
  * levsoi   (levsoi) float32 80B 0.01 0.04 0.09 0.16 ... 5.06 5.95 6.94 8.03


In [4]:
print(moisture_obs.depth.values, moisture_obs.depth.attrs)

[0.08 0.15] {}


In [ ]:


# ---------------------------------------------------------------------
# 1) Soil moisture, all data
# ---------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(6, 6))
rmse_all = scatter_with_rmse(moisture_obs, moisture_sim, "Soil Moisture (All Data)", ax=ax1)
fig1.tight_layout()
fig1.savefig("soil_moisture_rmse_all.png", dpi=150)
print(f"RMSE (all soil moisture): {rmse_all:.4f}")


# ---------------------------------------------------------------------
# 2) Soil moisture, only where soil temperature > 0 degC
# ---------------------------------------------------------------------
# Using OBSERVED soil temperature to define "above zero." If you'd rather
# filter using simulated temperature instead, swap in temp_sim here.
warm_mask = temp_obs > 0

moisture_obs_warm = moisture_obs.where(warm_mask)
moisture_sim_warm = moisture_sim.where(warm_mask)

fig2, ax2 = plt.subplots(figsize=(6, 6))
rmse_warm = scatter_with_rmse(
    moisture_obs_warm, moisture_sim_warm, "Soil Moisture (Soil Temp > 0°C)", ax=ax2
)
fig2.tight_layout()
fig2.savefig("soil_moisture_rmse_warm.png", dpi=150)
print(f"RMSE (soil moisture, temp > 0): {rmse_warm:.4f}")

plt.show()